# Percentile function for finding RFI outliers and comparing the data:

In [4]:
from chime.calibration import load_Learmonth_data
import numpy as np
import matplotlib.pyplot as plt
import chime
from scipy.optimize import curve_fit

sample_data = "/home/scratch/dbautist/CHIME_archive/2025_160"
chime_data, frequency, timestamps = chime.calibration.load_CHIME_data(sample_data)


df = load_Learmonth_data("/home/scratch/dbautist/CHIME_archive/learmonthData/L2515.SRD")
df

FileNotFoundError: /home/scratch/dbautist/CHIME_archive/learmonthData/L2515.SRD not found.

# Solar Frequency Calculations:

This code we are doing a few things.  One, Using the distance of the Earth from the sun during different times of the years along with the constant luminocity value of the sun in W/m^2 to calculate the solar flux and how it varies as the Earth makes its elliptical orbit around the Sun.

the Solar_Flux function uses the distance in meters of the Earth from the Sun in Summer, Winter, and the equinoxes and calculates the solar flux.  Percentile_difference uses the percent difference formula to calculate the percentile range between first, the average distance the Earth is from the Sun compared to when the Earth is prograde(Winter) and furtheest(Summer).  

In [35]:
#solar freq calculations for winter(peak):

'''
Values for this function:
-sees if the midan solar flux of that day at that specific season falls within the 7% range between 
winter and summer, if its more, potential outliers most likely exist
mid_flux = median flux of that day at that specific year
avg_flux = around what the average for that month should be'''

sun_lum = 4.827e26 #in Watts
winter_earth_dist = 1.417e11
avg_earth_dist = 1.49e11
summer_earth_dist = 1.52e11

def solar_flux(luminosity, distance):
    return luminosity/((4.0*np.pi)*(distance)**2)

summer_flux = solar_flux(sun_lum, summer_earth_dist)
winter_flux = solar_flux(sun_lum, winter_earth_dist)
average_flux = solar_flux(sun_lum, avg_earth_dist)

print(f'summer flux:{summer_flux}W/m^2\n winter flux:{winter_flux}W/m^2\n average flux: {average_flux}W/m^2')

'''Using these values, I'm able o calculate the overall percentile difference range for winter between 
average than average between summer.  When we compare the days, if the percentile range between sun flux
and difference depending on what season we are in.  If the range of the percent difference for 
a particular month is larger than what that average is, there is means for outliers/rfi happenings'''

def dif_percentile(mid_flux, avg_flux):
    return ((mid_flux-avg_flux)/avg_flux)*100

winter_vs_avg_per = dif_percentile(winter_flux, average_flux)
summer_vs_avg_per = dif_percentile(average_flux, summer_flux)
winter_vs_summer_per = winter_vs_avg_per + summer_vs_avg_per

print(f'percentile for winter vs average:{winter_vs_avg_per}%\n summer vs. average:{summer_vs_avg_per}%')
print(f'all together total percentile difference:{winter_vs_summer_per}%')


summer flux:1662.5712220926653W/m^2
 winter flux:1913.056225480041W/m^2
 average flux: 1730.1943838218522W/m^2
percentile for winter vs average:10.568861127283427%
 summer vs. average:4.0673843520562185%
all together total percentile difference:14.636245479339646%


This loop takes in a day of a certin file and sees if the flux falls within the percentile range or not, if it doesnt fall within the seasonal percentile assigned to it, then there is reason to suspect large ammounts of RFI or that the Sun was extra active that day

In [36]:
#if statement determining outliers:

today_flux_value = dif_percentile(np.nanmedian(df['410'])*10000, 350000) # Date:6/09/2025
print(today_flux_value)
print(np.nanmedian(df['410'])*10000)

'''For test run, we will assume summer and calculating the percents on that 
if there are any crazy outliers:'''

if -(summer_vs_avg_per) < today_flux_value < summer_vs_avg_per:
    print("Within the appropriate range")
else:
    print("Outlier exists here")

0.0
350000.0
Within the appropriate range


In [37]:
#If statement for winter now(using first a 'within range'date):

sample_data = "/home/scratch/dbautist/CHIME_archive/2025_312"
chime_data, frequency, timestamps = chime.calibration.load_CHIME_data(sample_data)
df = load_Learmonth_data("/home/scratch/dbautist/CHIME_archive/learmonthData/L241107.SRD")

flux = np.nanmedian(df['410'])*10000
print(f'flux value at this date:{flux}Jy')

today_flux = dif_percentile(np.nanmedian(df['410'])*10000, 440000)
print(f'Percent difference:{today_flux}%')

if -(winter_vs_avg_per) < today_flux < winter_vs_avg_per:
    print("Within the appropriate range")
else:
    print("Outlier exists here")


flux value at this date:590000.0Jy
Percent difference:34.090909090909086%
Outlier exists here


In [38]:
import glob
from tqdm import trange

learmonth = "/home/scratch/dbautist/CHIME_archive/learmonthData/"
learmonth_files = glob.glob(f'{learmonth}/L2508*SRD')


output_list = []

def learmonth_mid(learmonth_dat):
    return learmonth_dat.split()

for i in trange(len(learmonth_files)):     
    output = learmonth_mid(learmonth_files[i])
    output_list.append(output)  #appends output into a list 

    
#if statement:
    
if len(output_list) == 1:
    print(f'this list has enough values to be supported(items:{len(good_list)})')
else:
    print('does not have enough data')
     
    
print(output_list)    
type(output_list)

    

100%|██████████| 29/29 [00:00<00:00, 266743.02it/s]

does not have enough data
[['/home/scratch/dbautist/CHIME_archive/learmonthData/L250801.SRD'], ['/home/scratch/dbautist/CHIME_archive/learmonthData/L250802.SRD'], ['/home/scratch/dbautist/CHIME_archive/learmonthData/L250803.SRD'], ['/home/scratch/dbautist/CHIME_archive/learmonthData/L250804.SRD'], ['/home/scratch/dbautist/CHIME_archive/learmonthData/L250805.SRD'], ['/home/scratch/dbautist/CHIME_archive/learmonthData/L250806.SRD'], ['/home/scratch/dbautist/CHIME_archive/learmonthData/L250807.SRD'], ['/home/scratch/dbautist/CHIME_archive/learmonthData/L250808.SRD'], ['/home/scratch/dbautist/CHIME_archive/learmonthData/L250809.SRD'], ['/home/scratch/dbautist/CHIME_archive/learmonthData/L250810.SRD'], ['/home/scratch/dbautist/CHIME_archive/learmonthData/L250811.SRD'], ['/home/scratch/dbautist/CHIME_archive/learmonthData/L250812.SRD'], ['/home/scratch/dbautist/CHIME_archive/learmonthData/L250813.SRD'], ['/home/scratch/dbautist/CHIME_archive/learmonthData/L250814.SRD'], ['/home/scratch/dbaut

list

In [3]:
import glob

learmonth = "/home/scratch/dbautist/CHIME_archive/learmonthData/"
learmonth_files = glob.glob(f'{learmonth}/L25*SRD')
len(learmonth_files)

358

In [7]:
import glob
from tqdm import trange
import os
import datetime


learmonth = "/home/scratch/dbautist/CHIME_archive/learmonthData/"
learmonth_files = glob.glob(f'{learmonth}/L2501*SRD')
learmonth_files.sort()

print(learmonth_files[0])
my_path = learmonth_files[0]

df = load_Learmonth_data(my_path)
print(df)

median = np.nanmedian(df['410'])
print(median)

def daily_median(path, frequency):
    df = load_learmonth_data(path)
    med = np.nanmedian(df[frequency])
    return med


print(my_path[0:31])
    
#goal: median average day of data
#indexing path of lists to get to a single day





/home/scratch/dbautist/CHIME_archive/learmonthData/L250101.SRD
                           time  seconds   245   410   610   1415   2695  \
0     2025-01-01 00:00:00+00:00      0.0  20.0  43.0  75.0  154.0  210.0   
1     2025-01-01 00:00:01+00:00      1.0  20.0  43.0  75.0  153.0  211.0   
2     2025-01-01 00:00:02+00:00      2.0  20.0  43.0  75.0  155.0  210.0   
3     2025-01-01 00:00:03+00:00      3.0  20.0  43.0  75.0  155.0  212.0   
4     2025-01-01 00:00:04+00:00      4.0  20.0  43.0  75.0  156.0  211.0   
...                         ...      ...   ...   ...   ...    ...    ...   
86395 2025-01-01 23:59:55+00:00  86395.0  26.0  46.0  72.0  152.0  206.0   
86396 2025-01-01 23:59:56+00:00  86396.0  26.0  46.0  72.0  153.0  205.0   
86397 2025-01-01 23:59:57+00:00  86397.0  26.0  46.0  72.0  152.0  204.0   
86398 2025-01-01 23:59:58+00:00  86398.0  25.0  47.0  72.0  152.0  204.0   
86399 2025-01-01 23:59:59+00:00  86399.0  25.0  47.0  72.0  152.0  204.0   

        4975   8800  154

# Loop for mapping each Month with a Wild Card and calculating the Medians for each day:

This python block uses two deffinitions that do two very similar things, the only difference is the unit conversion, one in SFU and the other in Janskys.  The function takes in a path from the Learmonth data and iterates over the full list of each day and calculates the median for each day of that month, putting it in a list.  It also negates the days where CHIME produced more NaNs than would be liable to use for research purposes.  

In [8]:
import os, time


def daily_median(path, frequency):  #def in solar flux units
    
    '''Definition for mapping through each day in the wild card
    file path for the month(for example, January of 2026), maps out 
    and calculates the median for each day in that month in units 
    of Solar Flux Units'''

    df = load_Learmonth_data(path)
    med = np.nanmedian(df[frequency])
   
    return med

def daily_median_Jy(path, frequency):  #def in units of Janskys
    
    '''Same thing as the previous function except med is now being
    mulitplied by 10000 to convert SFU into units of Janksys'''

    df = load_Learmonth_data(path)
    med = np.nanmedian(df[frequency]) * 10000
    return med

    '''Median list = the list of each median for each day of the chosen
                    month in SFU
     Median List Jy = list of each median for each day of chosen month
                    in units of Janskys'''
                    
median_list = []  # defining empty lists 
median_list_Jy = []

for i in trange(len(learmonth_files)):     
    output = daily_median(learmonth_files[i], '410')
    output_Jy = daily_median_Jy(learmonth_files[i], '410')
    median_list.append(output)  
    median_list_Jy.append(output_Jy)
   
    
'''
df_410 = df['410']

column_NaN_count = df_410.isnull().sum()
print("NaN count per column:")
print(NaN_count)'''
    


100%|███████████████████████████████████████████| 29/29 [11:08<00:00, 23.05s/it]


'\ndf_410 = df[\'410\']\n\ncolumn_NaN_count = df_410.isnull().sum()\nprint("NaN count per column:")\nprint(NaN_count)'

In [9]:
#Loop for calculating NaNs in a Month:



for i in trange(len(learmonth_files)):
    output = daily_median(learmonth_files[i], '410')
    column_NaN_count = output.isnull().sum()
    
print(column_NaN_count)

  0%|                                                    | 0/29 [00:14<?, ?it/s]


AttributeError: 'numpy.float32' object has no attribute 'isnull'

This python code uses the os package to state at which time and date the file was made for organizational purposes with ti_c being the time that the file was last accessed and ti_m being when the file was originally made.

In [ ]:
import os, time


ti_c = os.path.getctime(my_path)
ti_m = os.path.getmtime(my_path)

c_ti = time.ctime(ti_c)
m_ti = time.ctime(ti_m)


print(f'the file located at the path {my_path} was created at {c_ti}, last modified {m_ti}')


In [ ]:
print(f'list in SFU:{median_list}\n list in Janskys:{median_list_Jy}')
print(f'outliers for this month: {column_NaN_count}') #For febuary: 40732
len(median_list)




In [ ]:
#plotting this plot:

plt.figure()

plt.plot(median_list)
plt.title(f"median vs each day during the month of {m_ti}")
plt.xlabel("Days")
plt.ylabel("Avg medians in Solar Flux Units for the month of: ")
monthly_median = np.nanmedian(median_list)

plt.hlines(monthly_median, xmin=0, xmax=30, color='red')




In [ ]:
#For both in SFU and Janskys:

fig, (ax1, ax2) = plt.subplots(nrows=2, ncols=1, figsize=(10,5), sharex='col') #for multiple graphs to exist

#first plot(in SFU):

ax1.plot(median_list, color='blue', label='SFU')

ax1.set_title(f"median vs each day during the month of {m_ti}")
ax1.set_ylabel("Avg medians in Solar Flux Units")
monthly_median = np.nanmedian(median_list)
ax1.hlines(monthly_median, xmin=0, xmax=30, color='red')

#Second plot(in Janskys):

ax2.plot(median_list_Jy, color='orange', label='Janskys')
ax2.set_xlabel("Days")
ax1.set_ylabel("Avg Medians in Janskys")
monthly_median_Jy = np.nanmedian(median_list_Jy)
ax2.hlines(monthly_median_Jy, xmin=0, xmax=30, color='purple')


In [ ]:
#count the numbber of NaN's present:

df_410 = df['410']

column_NaN_count = df_410.isnull().sum()
print("NaN count per column:")
print(column_NaN_count)


In [ ]:
#If statement for filtering out the days with no good data and printing them for people to know:

if column_NaN_count <= len(df_410):

In [ ]:
#If statement to see if the day is a good day:

len(df_410)

if column_NaN_count <= len(df_410):
 
    fig, (ax1, ax2) = plt.subplots(nrows=2, ncols=1, figsize=(10,5), sharex='col') #for multiple graphs to exist

    #first plot(in SFU):
    ax1.plot(median_list, color='blue', label='SFU')
    ax1.set_title(f"median vs each day during the month of {m_ti}")
    ax1.set_ylabel("Avg medians in Solar Flux Units")
    monthly_median = np.nanmedian(median_list)
    ax1.hlines(monthly_median, xmin=0, xmax=30, color='red')

    #Second plot(in Janskys):

    ax2.plot(median_list_Jy, color='orange', label='Janskys')
    ax2.set_xlabel("Days")
    ax1.set_ylabel("Avg Medians in Janskys")
    monthly_median_Jy = np.nanmedian(median_list_Jy)
    ax2.hlines(monthly_median_Jy, xmin=0, xmax=30, color='purple')
    print(f"WARNING: there are {column_NaN_count} non real numbers in this data set")
    
else:
    print(f"Too many non real numbers with total being {column_NaN_count}")